# Business CSV Data Validator

- Row-validation problem over a simplified 6-column CSV feed with strict business rules.
- The main behaviors are ordered rule evaluation, trimming, token-based word checks, and first-failure reporting.
- Important invariants are preserving rule order, using case-insensitive word matching, and handling stop words only in cross-column comparison.
- Similar patterns show up in ingestion pipelines, backoffice review systems, and partner-data compliance checks.
- Focus on empty fields, forbidden words, short business names, and overlap thresholds between descriptive columns.


In [ ]:
from typing import List


class Solution:
    def validateDatasetRows(self, data: List[str], forbiddenWords: List[str], ignoreWords: List[str]) -> List[str]:
        forbidden = {word.lower() for word in forbiddenWords}
        ignore = {word.lower() for word in ignoreWords}
        results = []

        for row in data[1:]:
            cols = row.split(',')
            trimmed = [col.strip() for col in cols]

            if any(value == '' for value in trimmed):
                results.append('INVALID:RULE_1')
                continue

            if not 5 <= len(trimmed[4]) <= 31:
                results.append('INVALID:RULE_2')
                continue

            col2_words = trimmed[1].lower().split()
            if any(word in forbidden for word in col2_words):
                results.append('INVALID:RULE_3')
                continue

            valid_col2 = [word for word in col2_words if word not in ignore]
            valid_target = {
                word
                for value in (trimmed[3], trimmed[4])
                for word in value.lower().split()
                if word not in ignore
            }

            if not valid_col2:
                results.append('INVALID:RULE_4')
                continue

            matches = sum(word in valid_target for word in valid_col2)
            if matches * 2 < len(valid_col2):
                results.append('INVALID:RULE_4')
                continue

            results.append('VALID')

        return results


In [ ]:
def test(solution):
    cases = [
        ((([
            'col1,col2,col3,col4,col5,col6',
            'id1,Land Water,c,d,land water llc,f',
            'id2,Good Company,c,d,land water,f',
            'id3,Apple banana,c,d,banana orange,f',
            'id4, ,c,d,hello world,f',
            'id5,LLC Inc,c,d,something real,f',
            'id6,blue sky ocean,c,d,green yellow pink,f',
            'id7,Best Group,c,d,hello group,f',
            'id8,land water,c,d,land,f',
        ], ['company', 'firm', 'co.', 'corporation', 'group'], ['llc', 'inc'])), ['VALID', 'INVALID:RULE_3', 'VALID', 'INVALID:RULE_1', 'INVALID:RULE_4', 'INVALID:RULE_4', 'INVALID:RULE_3', 'INVALID:RULE_2']),
        ((([
            'col1,col2,col3,col4,col5,col6',
            'id1,hello world,c,d,hello there,f',
        ], ['bad', 'wrong'], [])), ['VALID']),
        ((([
            'col1,col2,col3,col4,col5,col6',
            'id1,good name,c,d,test,f',
        ], [], [])), ['INVALID:RULE_2']),
        ((([
            'col1,col2,col3,col4,col5,col6',
            'id1,Alpha Beta,c,alpha,beta gamma,f',
            'id2,Alpha Beta Gamma,c,alpha,beta delta,f',
        ], [], [])), ['VALID', 'VALID']),
        ((([
            'col1,col2,col3,col4,col5,col6',
            'id1,Alpha Beta Gamma Delta,c,alpha,zzzzz,f',
        ], [], [])), ['INVALID:RULE_4']),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = solution(*args)
        assert got == expected, f'case {i}: expected {expected}, got {got}'


In [ ]:
def current_solution(data, forbiddenWords, ignoreWords):
    return Solution().validateDatasetRows(data, forbiddenWords, ignoreWords)

test(current_solution)
print('PASS')
